In [1]:
!pip install pyspark ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 14.4 MB/s  0:00:00


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Widgets") \
    .getOrCreate()

In [4]:
from google.colab import files, output


output.enable_custom_widget_manager()

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Number of records:", len(df))

display(df.head())

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k.csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [6]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Plant dropdown
plant_dropdown = widgets.Dropdown(
    options=create_options("plant_id", "All Plants"),
    description="Plant:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Product family dropdown
product_dropdown = widgets.Dropdown(
    options=create_options("product_family", "All Products"),
    description="Product Family:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 3. Quality status dropdown
quality_dropdown = widgets.Dropdown(
    options=create_options("quality_status", "All Statuses"),
    description="Quality:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 4. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [8]:
from IPython.display import display
from google.colab import output

output.enable_custom_widget_manager()

display(widgets.VBox([
    plant_dropdown,
    product_dropdown,
    quality_dropdown,
    units_slider
]))

In [16]:
from google.colab import files, output

output.enable_custom_widget_manager()

import math
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyspark.sql import functions as F

uploaded = files.upload()

file_name = next(iter(uploaded))

df = pd.read_csv(file_name)                                       # for building the widgets
sdf = spark.read.csv(file_name, header=True, inferSchema=True)    # for filtering with PySpark

print("Dataset loaded successfully!")
print("Number of records:", len(df))

display(df.head())

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k (1).csv
Dataset loaded successfully!
Number of records: 12500


,production_id,production_date,plant_id,machine_id,operator_id,shift,product_code,product_family,units_planned,units_produced,defective_units,downtime_minutes,energy_kwh,temperature_c,vibration_mm_s,quality_status,maintenance_flag,supervisor
0,PR0120001,2026-06-01,PL05,MC024,OP0191,Evening,PMP-210,Pumps,293,292,10,44.0,120.42,70.59,2.21,Pass,Yes,Anita
1,PR0120002,2026-09-22,PL01,MC001,OP0461,Night,MTR-100,Motors,1009,897,25,59.0,317.23,81.74,3.03,Pass,No,Anita
2,PR0120003,2026-05-24,PL03,MC012,OP0451,Evening,MTR-100,Motors,960,983,34,0.0,350.82,67.29,2.01,Pass,No,Deepa
3,PR0120004,2026-09-12,PL04,MC016,OP0237,Night,CTL-130,Controllers,310,320,15,14.0,142.34,59.70,3.48,Pass,No,Anita
4,PR0120005,2026-02-04,PL04,MC018,OP0072,Morning,VLV-220,Valves,957,979,22,14.0,296.70,62.52,2.33,Pass,No,Harini


In [17]:
# Function to create dropdown options
def create_options(column, all_label):
    values = sorted(df[column].dropna().astype(str).unique())

    return [(all_label, "__ALL__")] + [
        (value, value) for value in values
    ]


# 1. Shift dropdown
shift_dropdown = widgets.Dropdown(
    options=create_options("shift", "All Shifts"),
    description="Shift:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 2. Machine dropdown
machine_dropdown = widgets.Dropdown(
    options=create_options("machine_id", "All Machines"),
    description="Machine:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 3. Supervisor dropdown
supervisor_dropdown = widgets.Dropdown(
    options=create_options("supervisor", "All Supervisors"),
    description="Supervisor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px")
)


# 4. Maintenance flag dropdown
maintenance_dropdown = widgets.Dropdown(
    options=create_options("maintenance_flag", "All"),
    description="Maintenance:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="260px")
)


# 5. Slider for units produced
units = pd.to_numeric(df["units_produced"], errors="coerce").dropna()

units_slider = widgets.IntRangeSlider(
    value=(int(units.min()), int(units.max())),
    min=int(units.min()),
    max=int(units.max()),
    step=1,
    description="Units Produced:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)


# 6. Slider for energy consumed (kWh)
energy = pd.to_numeric(df["energy_kwh"], errors="coerce").dropna()
energy_min = math.floor(energy.min())
energy_max = math.ceil(energy.max())

energy_slider = widgets.FloatRangeSlider(
    value=(energy_min, energy_max),
    min=energy_min,
    max=energy_max,
    step=1.0,
    description="Energy (kWh):",
    continuous_update=False,
    readout_format=".0f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="90%")
)

In [20]:
display(widgets.VBox([
    shift_dropdown,
    machine_dropdown,
    supervisor_dropdown,
    maintenance_dropdown,
    units_slider,
    energy_slider
]))

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, desc

spark = SparkSession.builder \
    .appName("LazyExecutionDemo") \
    .getOrCreate()

data = [
    (1, "Hyderabad", "Laptop", 2, 55000),
    (2, "Bangalore", "Mobile", 5, 20000),
    (3, "Chennai", "Tablet", 3, 15000),
    (4, "Hyderabad", "Mobile", 4, 22000),
    (5, "Mumbai", "Laptop", 1, 60000),
    (6, "Bangalore", "Tablet", 6, 12000),
    (7, "Chennai", "Laptop", 2, 58000),
    (8, "Mumbai", "Mobile", 8, 18000),
    (9, "Hyderabad", "Tablet", 5, 14000),
    (10, "Bangalore", "Laptop", 3, 52000),
    (11, "Mumbai", "Tablet", 7, 13000),
    (12, "Chennai", "Mobile", 6, 21000)
]

columns = ["sale_id", "city", "product", "quantity", "price"]

df = spark.createDataFrame(data, columns)

df.show()

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      2|Bangalore| Mobile|       5|20000|
|      3|  Chennai| Tablet|       3|15000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      6|Bangalore| Tablet|       6|12000|
|      7|  Chennai| Laptop|       2|58000|
|      8|   Mumbai| Mobile|       8|18000|
|      9|Hyderabad| Tablet|       5|14000|
|     10|Bangalore| Laptop|       3|52000|
|     11|   Mumbai| Tablet|       7|13000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+



In [23]:
from pyspark.sql.functions import col

filtered_df = df.filter(col("quantity") > 3)

print("Transformation created")


selected_df = filtered_df.select(
    "city", "product", "quantity"
)

print("Another transformation created")


selected_df.show()

Transformation created
Another transformation created
+---------+-------+--------+
|     city|product|quantity|
+---------+-------+--------+
|Bangalore| Mobile|       5|
|Hyderabad| Mobile|       4|
|Bangalore| Tablet|       6|
|   Mumbai| Mobile|       8|
|Hyderabad| Tablet|       5|
|   Mumbai| Tablet|       7|
|  Chennai| Mobile|       6|
+---------+-------+--------+



In [24]:



step1 = df.filter(col("quantity") >= 3)


step2 = step1.withColumn(
    "revenue",
    col("quantity") * col("price")
)


step3 = step2.groupBy("city").agg(
    sum("revenue").alias("total_revenue")
)


step4 = step3.orderBy(desc("total_revenue"))

print("All transformations defined")


step4.show()

All transformations defined
+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|       328000|
|   Mumbai|       235000|
|  Chennai|       171000|
|Hyderabad|       158000|
+---------+-------------+



In [25]:
result = df.filter(col("price") > 20000) \
           .select("city", "product", "price") \
           .orderBy(desc("price"))

result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [price#39L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(price#39L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=140]
      +- Project [city#36, product#37, price#39L]
         +- Filter (isnotnull(price#39L) AND (price#39L > 20000))
            +- Scan ExistingRDD[sale_id#35L,city#36,product#37,quantity#38L,price#39L]




In [26]:
result.show()

+---------+-------+-----+
|     city|product|price|
+---------+-------+-----+
|   Mumbai| Laptop|60000|
|  Chennai| Laptop|58000|
|Hyderabad| Laptop|55000|
|Bangalore| Laptop|52000|
|Hyderabad| Mobile|22000|
|  Chennai| Mobile|21000|
+---------+-------+-----+



In [27]:


high_value_sales = df.filter(col("price") > 20000)


high_value_sales.show()


print("Total records:", high_value_sales.count())

records = high_value_sales.collect()
print(records)

+-------+---------+-------+--------+-----+
|sale_id|     city|product|quantity|price|
+-------+---------+-------+--------+-----+
|      1|Hyderabad| Laptop|       2|55000|
|      4|Hyderabad| Mobile|       4|22000|
|      5|   Mumbai| Laptop|       1|60000|
|      7|  Chennai| Laptop|       2|58000|
|     10|Bangalore| Laptop|       3|52000|
|     12|  Chennai| Mobile|       6|21000|
+-------+---------+-------+--------+-----+

Total records: 6
[Row(sale_id=1, city='Hyderabad', product='Laptop', quantity=2, price=55000), Row(sale_id=4, city='Hyderabad', product='Mobile', quantity=4, price=22000), Row(sale_id=5, city='Mumbai', product='Laptop', quantity=1, price=60000), Row(sale_id=7, city='Chennai', product='Laptop', quantity=2, price=58000), Row(sale_id=10, city='Bangalore', product='Laptop', quantity=3, price=52000), Row(sale_id=12, city='Chennai', product='Mobile', quantity=6, price=21000)]


In [1]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv

File: lazy_execution_stores (1).csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+---

In [2]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000 (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Comple

In [3]:

completed_df = sales_df.filter(
    F.col("order_status") == "Completed"
)

print("Transformation created")


print("Completed Orders:", completed_df.count())

completed_df.show(10)

Transformation created
Completed Orders: 21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04

In [4]:

completed_df.cache()

print("Is Cached:", completed_df.is_cached)

completed_df.count()

completed_df.show(10)

completed_df.groupBy("product_id").count().show()

Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       

Persist

In [5]:
high_quantity_df = sales_df.filter(
    F.col("quantity") > 5
)

high_quantity_df.persist(
    StorageLevel.MEMORY_AND_DISK
)

high_quantity_df.count()

high_quantity_df.show(10)

print(high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

Unpersist

In [6]:
print("Before:", completed_df.is_cached)

completed_df.unpersist(blocking=True)

print("After:", completed_df.is_cached)

high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [8]:

 #Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [9]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")


First Execution: 1.091 seconds
Second Execution: 0.088 seconds


Partitioning


In [10]:
partitioned_df = sales_df.repartition(8)

print("Original Partitions:",
      sales_df.rdd.getNumPartitions())

print("New Partitions:",
      partitioned_df.rdd.getNumPartitions())

partitioned_df.count()

Original Partitions: 1
New Partitions: 8


25000

In [11]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [13]:
store_partitioned_df = sales_df.repartition(
    8,
    "store_id"
)

print("Partitions:",
      store_partitioned_df.rdd.getNumPartitions())

store_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy(
    "partition_id", "store_id"
).count().orderBy(
    "partition_id", "store_id"
).show()

Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



In [14]:
df8 = sales_df.repartition(8)

df_repartition = df8.repartition(3)

df_coalesce = df8.coalesce(3)

print("Repartition:",
      df_repartition.rdd.getNumPartitions())

print("Coalesce:",
      df_coalesce.rdd.getNumPartitions())

print("REPARTITION PLAN")
df_repartition.explain()

print("COALESCE PLAN")
df_coalesce.explain()

Repartition: 3
Coalesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1101]
         +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1101]
   +- FileScan csv [order_id#64,order_date#65,store_id#66,product_id#67,quantity#68,unit_price#69,discount_pct#70,payment_method#71,order_status#72,sales_channel#73] Batched: false, DataFilters: []

Exercise


In [16]:
df12 = sales_df.repartition(12)

print("Partitions:", df12.rdd.getNumPartitions())

Partitions: 12


In [17]:
df12.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show(12)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [18]:
df_product = sales_df.repartition(6, "product_id")

print("Partitions:", df_product.rdd.getNumPartitions())

df_product.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Partitions: 6
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 6562|
|           1| 3419|
|           2| 5077|
|           3| 6548|
|           4| 3394|
+------------+-----+



In [19]:
df4 = df12.coalesce(4)

print("Partitions:", df4.rdd.getNumPartitions())

df4.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

Partitions: 4
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 6250|
|           1| 6250|
|           2| 6250|
|           3| 6250|
+------------+-----+

